# AND gate using Adaline 

#### Formal Definiton:

**ADALINE (Adaptive Linear Neuron)** is a single-layer neural network model developed by *Bernard Widrow and Ted Hoff*. 
Unlike the basic Perceptron, which updates weights based on the final thresholded output (0 or 1), ADALINE uses a **Linear Activation Function** to update weights based on the continuous net input before it is thresholded. 
This allows it to **minimize the Mean Squared Error (MSE)** using a process called **Gradient Descent**.

#### Task
This task involves training a mathematical model to find a decision boundary (a line) that separates the "1" case from the "0" cases.

The key difference from a Perceptron: ADALINE calculates how far it was from the right answer (the error) using a continuous value, making the learning process smoother.

---

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.animation import FuncAnimation
from IPython.display import HTML



### 1. Dataset (AND Gate)
Inputs: [x1, x2], Targets: y
| Input $x_1$ | Input $x_2$ | Target ($y$) |
| :---: | :---: | :---: |
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

In [5]:
X = np.array([[0,0], [0,1], [1,0], [1,1]])
y = np.array([0, 0, 0, 1])

### 2. Setup ADALINE Parameters
Low learning rate to see movement.
Initialize weights slightly offset from zero to see the initial line.
eta - Learning rate

In [6]:
weights = np.array([0.1, 0.2])
bias = -0.3
eta = 0.05  
epochs = 30

history = []
history.append((weights.copy(), bias))


### 3. Running the Training


In [12]:
epoch_data = []

for epoch in range(epochs):
    for i in range(len(X)):
        # Calculate Net Input (Linear Activation)
        net_input = np.dot(X[i], weights) + bias
        
        # Calculate Continuous Error (Widrow-Hoff Rule)
        error = y[i] - net_input
        
        # Update Weights and Bias (Gradient Descent)
        weights += eta * error * X[i]
        bias += eta * error
        
        # Save this state
        history.append((weights.copy(), bias))

        total_epoch_error = 0
    
    for i in range(len(X)):
        net_input = np.dot(X[i], weights) + bias
        error = y[i] - net_input
        
        weights += eta * error * X[i]
        bias += eta * error
        
        total_epoch_error += error**2 # Sum of Squared Errors
        history.append((weights.copy(), bias))

    epoch_data.append({
        "Epoch": epoch,
        "Weights": weights.copy(),
        "Bias": f"{bias:.4f}",
        "MSE": f"{(total_epoch_error / len(X)):.4f}"
    })

df_results = pd.DataFrame(epoch_data)
display(df_results)

,Epoch,Weights,Bias,MSE
0,0,"[0.5258263873116437, 0.5126632679834425]",-0.2625,0.0692
1,1,"[0.5258575484501842, 0.512694479884208]",-0.2626,0.0693
2,2,"[0.5258867204903025, 0.5127237274829326]",-0.2626,0.0693
3,3,"[0.5259140308998493, 0.5127511338770147]",-0.2626,0.0693
4,4,"[0.525939598928679, 0.5127768145024798]",-0.2627,0.0693
5,5,"[0.5259635361433593, 0.5128008776054769]",-0.2627,0.0693
6,6,"[0.5259859469266092, 0.5128234246852983]",-0.2627,0.0693
7,7,"[0.5260069289438398, 0.5128445509105916]",-0.2628,0.0693
8,8,"[0.5260265735790064, 0.5128643455103326]",-0.2628,0.0693
9,9,"[0.5260449663418278, 0.5128828921410464]",-0.2628,0.0693



### 4. Setting up the "Canvas"

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

# Plot the static data points once
# Mask the data: Red (Class 0) / Green (Class 1)
class0_mask = (y == 0)
class1_mask = (y == 1)

ax.scatter(X[class0_mask, 0], X[class0_mask, 1], c='red', marker='o', s=200, label='Class 0 (Target 0)')
ax.scatter(X[class1_mask, 0], X[class1_mask, 1], c='green', marker='X', s=200, label='Class 1 (Target 1)')

# Setup plot boundaries (add some padding)
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('$x_1$ Input', fontsize=12)
ax.set_ylabel('$x_2$ Input', fontsize=12)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.1), ncol=2)
ax.grid(True, linestyle='--', alpha=0.5)

# Initialize an empty line object that will be updated in the animation
line, = ax.plot([], [], 'b-', lw=3, label='Decision Boundary')

# Text object to display epoch/update info
text_step = ax.text(0.05, 0.05, '', transform=ax.transAxes)

### 5. The Animation Function

In [ ]:
def animate(i):
    # i is the current update step index (from history list)
    current_w, current_b = history[i]
    
    w1, w2 = current_w
    
    # We define a set of x-values to draw our line across
    x_coords = np.linspace(-0.5, 1.5, 10)
    
    # Prevent division by zero if w2 is tiny
    if abs(w2) < 1e-6:
        w2 = 1e-6

    # Calculate y-coordinates (x2) using our rearranged line equation:
    # x2 = (Threshold - bias - w1*x1) / w2
    # Threshold is 0.5 (midpoint between 0 and 1)
    y_coords = (0.5 - current_b - (w1 * x_coords)) / w2
    
    # Update the line data
    line.set_data(x_coords, y_coords)
    
    # Update the text readout
    epoch_num = i // 4 # Integer division (4 updates per epoch)
    sample_num = i % 4
    text_step.set_text(f'Epoch: {epoch_num} | Sample Update: {sample_num}\nWeights: [{w1:.3f}, {w2:.3f}] | Bias: {current_b:.3f}')
    
    return line, text_step

### 6. Generate and Display the Animation

In [ ]:
# 'frames' tells FuncAnimation how many times to call 'animate'
anim = FuncAnimation(fig, animate, frames=len(history), interval=200, blit=True)

# Important for Jupyter: close the static plot so only the animation shows
plt.close(fig)

# Render the animation in the notebook as HTML5 video
HTML(anim.to_html5_video())